In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

c:\Users\adnan\anaconda3\envs\tf-gpu\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
model = load_model('model_news_claf.h5')
with open('tokenizer.pkl', 'rb') as file:
    tokenizer = pickle.load(file)

In [4]:
max_len = 64

In [5]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
def preprocess_text(text):

    text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', "", text)
    text = re.sub(r'@\w+', "", text)
    text = re.sub(r'[^a-zA-Z\s]', "", text)
    text = re.sub(r'\d+', "", text)
    text = text.lower()
    text = re.sub(r'\s+', " ", text).strip()
    tokens = word_tokenize(text)

    processed_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    processsed_text = ' '.join(processed_tokens)
    return processsed_text
    

In [16]:
def predict_news(news_text):
    processed_text = preprocess_text(news_text)
    # print(processed_text)

    sequence = tokenizer.texts_to_sequences([processed_text])
    # print(sequence)

    padded_sequence=pad_sequences(sequence,maxlen=max_len,padding='post')
    # print(padded_sequence)

    prediction = model.predict(padded_sequence)[0][0]

    if prediction >= 0.75:

        result = "FAKE NEWS"

    elif prediction <= 0.25:

        result = "REAL NEWS"

    else:

        result = "UNCERTAIN"
    
    print(f"\nPrediction Score: {prediction:.4f}")

    print(f"\nPrediction: {result}")


In [17]:
example_news_1 = """Title: Rob Jetten Sworn in as Dutch PM

Body: Following complex European coalition negotiations, Rob Jetten was officially sworn in as the Prime Minister of the Netherlands. His historic appointment marks a significant milestone in global governance, making him the youngest and first openly gay leader to hold the nation's highest political office."""

predict_news(example_news_1)

1/1 [==============================] - 0s 30ms/step

Prediction Score: 0.1099

Prediction: REAL NEWS


In [19]:
example_news_1 = """ Title: Billionaire Leaves Fortune to Backyard Raccoons

Body: A California court upheld a controversial will leaving a $45 million estate to a wild colony of backyard raccoons. The tech entrepreneur directed his liquid wealth and mansion into a specialized wildlife trust, creating a historic legal precedent for wild animal inheritance."""

predict_news(example_news_1)

1/1 [==============================] - 0s 37ms/step

Prediction Score: 0.9798

Prediction: FAKE NEWS
